# Task 4.1 — Red Marker Detection

**Platform:** Zybo Legacy / PYNQ 3.0.1, OpenCV 4.5.4, Logitech C270 (`/dev/video0`).

**Objective:** capture one frame and detect a saturated red marker using HSV segmentation and red-channel dominance. The aspect-ratio filter deliberately targets a **vertical marker**; this is not general red-object recognition.

Place a vertical red marker in view and run the single self-contained code cell. The camera is always released. The output includes the original image, binary mask and annotated detection.

In [ ]:
import cv2
import numpy as np
import time
from IPython.display import display, Image

DEVICE = "/dev/video0"
cap = cv2.VideoCapture(DEVICE, cv2.CAP_V4L2)
try:
    if not cap.isOpened():
        print("ERROR: Cannot open camera:", DEVICE)
    else:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        frame = None
        # Warm up automatic exposure before capturing the test image.
        for _ in range(30):
            ret, img = cap.read()
            if ret and img is not None:
                frame = img
            time.sleep(0.05)
        if frame is None:
            print("ERROR: No frame captured.")
        else:
            print("Frame shape:", frame.shape)
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            # Combine the two HSV hue intervals used for red.
            hsv_mask = cv2.bitwise_or(
                cv2.inRange(hsv, (0, 100, 50), (10, 255, 255)),
                cv2.inRange(hsv, (170, 100, 50), (179, 255, 255))
            )
            b, g, r = [c.astype(np.int16) for c in cv2.split(frame)]
            # Signed conversion avoids unsigned overflow in channel differences.
            dominant = (r > g + 45) & (r > b + 45) & (r > 80)
            mask = cv2.bitwise_and(hsv_mask, dominant.astype(np.uint8) * 255)
            kernel = np.ones((3, 3), dtype=np.uint8)
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
            contours, _ = cv2.findContours(
                mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            result = frame.copy()
            count = 0
            for contour in contours:
                if cv2.contourArea(contour) < 80:
                    continue
                x, y, w, h = cv2.boundingRect(contour)
                # Marker-specific filter: assume the colored region is vertical.
                if h / float(max(w, 1)) < 1.5:
                    continue
                cv2.rectangle(result, (x, y), (x+w, y+h), (0, 0, 255), 2)
                cv2.putText(result, "RED", (x, max(15, y-8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                count += 1
            print("Red regions detected:", count)
            # Show original frame, binary mask, and annotated result.
            mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
            panel = np.hstack([
                cv2.resize(im, (320, 240))
                for im in (frame, mask_bgr, result)
            ])
            ok, jpg = cv2.imencode(".jpg", panel)
            if ok:
                display(Image(data=jpg.tobytes()))
finally:
    # Release the camera on success and on errors.
    cap.release()
    print("Camera released.")

## Verification and interpretation

- With a vertical red marker and the tested lighting, one marker region was detected without false detections on the hand.
- A mask represents **red pixels/regions**, not necessarily the full physical marker.
- Turn the marker sideways and observe how the marker-specific height/width filter changes detection.
- Lighting, exposure and different red objects may require recalibration. The thresholds are experimental, not universal.